# 📓 Notebook B — Domain Controls & Steps Curve 

This version is adapted for Kaggle notebooks instead of Colab/Google Drive.


## Kaggle quick-start

1. Turn on **GPU** and **Internet** in the Kaggle notebook settings.
2. Add a Kaggle input dataset that contains `E00_baseline.csv`.
3. In **Cell 1**, choose only the experiments you want to run in this session.
4. Optional: add a previous notebook-output dataset as input if you want to seed `all_results.csv` or resume from saved checkpoints.
5. After a run finishes, click **Save Version** so the outputs under `/kaggle/working/lora_forgetting_research/` can be reused later.

This notebook is set up to auto-discover the baseline CSV, save checkpoints during training, resume from saved checkpoints when available, and skip experiments that are already present in `all_results.csv`.


## Step 0 — Install Dependencies

> **Why `git+` for transformers?**  
> `Qwen/Qwen3.5-9B` uses a new architecture (`model_type: qwen3_5`) whose mapping class `Qwen3_5ForCausalLM` does not yet exist in any PyPI release of `transformers`. Installing from the `main` branch is the only way to load this model. Once a release ships with Qwen3.5 support, swap the git URL for a `>=X.Y.Z` pin for reproducibility.
>
> Run once per Colab session. *Runtime → Run all* handles this automatically.


In [ ]:
# --- CELL 0: Install ---
import subprocess, sys

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# Qwen3.5 architecture (qwen3_5) is not in any PyPI release yet (as of Mar 2026).
# We must install transformers from the git main branch to get Qwen3_5ForCausalLM.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/huggingface/transformers.git"
])

pip("peft>=0.12.0", "trl>=0.10.0",
    "bitsandbytes>=0.43.0", "accelerate>=0.33.0",
    "datasets>=2.20.0", "sentence-transformers>=3.0.0",
    "scipy", "scikit-learn", "matplotlib", "seaborn")

print("✅ Dependencies installed")

## Step 1 — Kaggle Config & Shared Settings


In [ ]:
# --- CELL 1: Kaggle config + shared settings ---
import os, glob, shutil, torch, random, re
import numpy as np
import pandas as pd
from datetime import datetime

KAGGLE_INPUT_ROOT = "/kaggle/input"
KAGGLE_WORKING_ROOT = "/kaggle/working"

# Add a Kaggle input dataset that contains E00_baseline.csv.
BASELINE_FILENAME = "E00_baseline.csv"
BASELINE_CSV_OVERRIDE = None  # e.g. "/kaggle/input/my-baseline-dataset/E00_baseline.csv"

# If you attach a previous notebook-output dataset as input, the notebook can reuse it.
AUTO_SEED_RESULTS_FROM_INPUT = True
RESULTS_SEED_CSV_OVERRIDE = None  # e.g. "/kaggle/input/my-previous-output/lora_forgetting_research/results/all_results.csv"
SEARCH_PREVIOUS_OUTPUTS = True
SKIP_COMPLETED_EXPERIMENTS = True

# Kaggle run controls — strongly recommended: run only a small set per session.
RUN_DOMAIN_CONTROL_EXPERIMENTS = ["E05"]   # choose from ["E05", "E05b"]
RUN_STEPS_EXPERIMENTS = []                 # e.g. [("E06", 100)] or [("E08", 1000)]

# Use smaller values for a smoke test.
DOMAIN_DATASET_SIZE = 4000
MEDQA_DATASET_SIZE = 4000
MMLU_MAX_SAMPLES = None                    # e.g. 20 for a quick test

DRIVE_BASE  = f"{KAGGLE_WORKING_ROOT}/lora_forgetting_research"
RESULTS_CSV = f"{DRIVE_BASE}/results/all_results.csv"

def find_file_by_name(filename, roots=(KAGGLE_INPUT_ROOT, KAGGLE_WORKING_ROOT)):
    matches = []
    for root in roots:
        if not os.path.exists(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            if filename in filenames:
                matches.append(os.path.join(dirpath, filename))
    return sorted(matches)

def _results_priority(path):
    norm = path.replace("\\", "/")
    score = 0
    if "/lora_forgetting_research/results/" in norm:
        score += 10
    if norm.startswith(KAGGLE_INPUT_ROOT):
        score += 5
    return score

baseline_matches = []
if BASELINE_CSV_OVERRIDE and os.path.exists(BASELINE_CSV_OVERRIDE):
    BASELINE_CSV = BASELINE_CSV_OVERRIDE
else:
    baseline_matches = find_file_by_name(BASELINE_FILENAME, roots=(KAGGLE_INPUT_ROOT,))
    if not baseline_matches:
        baseline_matches = find_file_by_name(BASELINE_FILENAME)
    BASELINE_CSV = baseline_matches[0] if baseline_matches else os.path.join(KAGGLE_INPUT_ROOT, BASELINE_FILENAME)

os.makedirs(f"{DRIVE_BASE}/results", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/adapters", exist_ok=True)

if AUTO_SEED_RESULTS_FROM_INPUT and not os.path.exists(RESULTS_CSV):
    if RESULTS_SEED_CSV_OVERRIDE and os.path.exists(RESULTS_SEED_CSV_OVERRIDE):
        shutil.copy2(RESULTS_SEED_CSV_OVERRIDE, RESULTS_CSV)
        print(f"♻️ Seeded results CSV from override: {RESULTS_SEED_CSV_OVERRIDE}")
    else:
        prior_result_matches = find_file_by_name("all_results.csv", roots=(KAGGLE_INPUT_ROOT,))
        prior_result_matches = sorted(prior_result_matches, key=_results_priority, reverse=True)
        if prior_result_matches:
            shutil.copy2(prior_result_matches[0], RESULTS_CSV)
            print(f"♻️ Seeded results CSV from: {prior_result_matches[0]}")

# Qwen3.5 series. No -Instruct suffix — all weights are instruct.
MODEL_8B     = "Qwen/Qwen3.5-9B"
SEED_PRIMARY = 42
SEED_REPEAT  = 7

MMLU_SUBJECTS = [
    "abstract_algebra","anatomy","astronomy","business_ethics",
    "clinical_knowledge","college_biology","college_chemistry",
    "college_computer_science","college_mathematics","college_medicine",
    "college_physics","computer_security","conceptual_physics",
    "econometrics","electrical_engineering","elementary_mathematics",
    "formal_logic","global_facts","high_school_biology",
    "high_school_chemistry","high_school_computer_science",
    "high_school_european_history","high_school_geography",
    "high_school_government_and_politics","high_school_macroeconomics",
    "high_school_mathematics","high_school_microeconomics",
    "high_school_physics","high_school_psychology","high_school_statistics",
    "high_school_us_history","high_school_world_history","human_aging",
    "human_sexuality","international_law","jurisprudence","logical_fallacies",
    "machine_learning","management","marketing","medical_genetics",
    "miscellaneous","moral_disputes","moral_scenarios","nutrition",
    "philosophy","prehistory","professional_accounting","professional_law",
    "professional_medicine","professional_psychology","public_relations",
    "security_studies","sociology","us_foreign_policy","virology",
    "world_religions"
]
MEDICAL_MMLU = {
    "anatomy","clinical_knowledge","college_biology","college_medicine",
    "high_school_biology","medical_genetics","professional_medicine",
    "virology","human_aging"
}
DOMAIN_DESCRIPTIONS = {
    "medqa":      "medicine clinical knowledge anatomy pharmacology pathology diagnosis treatment disease symptoms",
    "gsm8k":      "mathematics arithmetic algebra word problems numerical computation",
    "codealpaca": "programming code software functions algorithms debugging python",
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED_PRIMARY)
print(f"✅ Kaggle config ready | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"   Working dir: {DRIVE_BASE}")
print(f"   Results CSV: {RESULTS_CSV}")
print(f"   Baseline path: {BASELINE_CSV}")
print(f"   Baseline exists: {os.path.exists(BASELINE_CSV)}")
if baseline_matches:
    print("   Baseline candidates:")
    for path in baseline_matches[:5]:
        print(f"      - {path}")
print(f"   Domain experiments this session: {RUN_DOMAIN_CONTROL_EXPERIMENTS}")
print(f"   Steps experiments this session: {RUN_STEPS_EXPERIMENTS}")
print(f"   MMLU max samples: {MMLU_MAX_SAMPLES}")


## Step 2 — Load E00 Baseline from Notebook A

### Why this notebook depends on Notebook A's baseline

All forgetting metrics are **deltas relative to the unmodified baseline**:

$$\text{forgetting}_{\text{subj}} = \text{acc}_{\text{post-finetune}} - \text{acc}_{\text{baseline}}$$

Without `E00_baseline.csv`, there is no reference point and no meaningful result. The `FileNotFoundError` below is intentional — it prevents silent failures where forgetting is computed against stale or wrong baselines.

### When can Notebook B start running?

Notebook A's E00 (~2 hours) must complete before Cell 2 here will pass. The rest of Notebook B (E05, E05b, E06–E08) can then run in **parallel** on Account 2 while Notebook A continues with E01–E04 on Account 1. The two notebooks write to the **same Drive CSV** but different `exp_id` rows, so there is no write conflict.


In [ ]:
# --- CELL 2: Load E00 baseline (must be added as a Kaggle input) ---
if not os.path.exists(BASELINE_CSV):
    raise FileNotFoundError(
        f"\n❌ Baseline not found: {BASELINE_CSV}\n"
        f"   Add a Kaggle input dataset containing E00_baseline.csv, or set BASELINE_CSV_OVERRIDE in Cell 1.\n"
        f"   Checked under /kaggle/input and /kaggle/working."
    )

baseline_df   = pd.read_csv(BASELINE_CSV)
baseline_accs = dict(zip(baseline_df["subject"], baseline_df["accuracy"]))
print(f"✅ Baseline loaded: {len(baseline_accs)} subjects")
print(f"   Mean baseline accuracy: {np.nanmean(list(baseline_accs.values())):.3f}")
print(f"   Medical subjects mean:  {np.nanmean([baseline_accs[s] for s in MEDICAL_MMLU if s in baseline_accs]):.3f}")


## Step 3 — Shared Utility Functions

These mirror the utilities from Notebook A with one important update: the MMLU evaluator now disables Qwen3.5's **extended thinking mode** during evaluation.

### Why `enable_thinking=False` is critical for MMLU evaluation

Qwen3.5 models can emit a `<think>...</think>` block of chain-of-thought reasoning before answering. This is valuable for hard reasoning but **breaks multiple-choice evaluation** because:

1. The `<think>` block can span hundreds of tokens — with `max_new_tokens=5`, the model exhausts its budget inside the reasoning trace and never outputs an answer letter
2. Even with more tokens, `re.search(r"[ABCD]", gen_text)` would match letters *inside* the thinking trace rather than the final answer

We pass `enable_thinking=False` to `apply_chat_template`. A `TypeError` fallback handles older builds where this kwarg is not yet recognised, by appending the ` /no_think` control token to the prompt text instead.

> **Training vs evaluation asymmetry:** `enable_thinking=False` applies *only during MMLU eval*. The `run_finetune` function leaves thinking fully enabled — SFT trains on plain text completions and the thinking flag is a generation-time behaviour that does not alter the training gradient.

### Semantic proximity functions

`compute_subject_similarities` encodes each MMLU subject name and the domain description into the `all-MiniLM-L6-v2` embedding space, then computes cosine similarity. This gives a scalar per-subject "closeness" score to the training domain.

`run_proximity_test` runs two statistical tests:
- **H1 (Pearson r):** does forgetting correlate continuously with cosine similarity across all 57 subjects?
- **H3 (t-test):** do the 9 *a priori* proximal subjects (medical MMLU) forget more than the 48 distal ones?

For E05 (GSM8K) and E05b (CodeAlpaca), the *same* proximity functions are applied — but with math/code domain descriptions. H1 is evaluated against the appropriate domain embedding, while H3 always tests the `MEDICAL_MMLU` partition (giving us a fixed reference group to compare across domains).


In [ ]:
# --- CELL 3: Shared utility functions (Kaggle-friendly) ---
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, ttest_ind

# ── Logger / results helpers ──────────────────────────────────────────────────
def log_result(exp_id, result_dict):
    row = {"exp_id": exp_id, "timestamp": datetime.now().isoformat(), **result_dict}
    df_new = pd.DataFrame([row])
    if os.path.exists(RESULTS_CSV):
        df_new.to_csv(RESULTS_CSV, mode="a", header=False, index=False)
    else:
        df_new.to_csv(RESULTS_CSV, index=False)
    print(f"  💾 {exp_id} → saved")

def experiment_already_logged(exp_id):
    if not os.path.exists(RESULTS_CSV):
        return False
    try:
        df = pd.read_csv(RESULTS_CSV, usecols=["exp_id"])
    except Exception:
        return False
    return exp_id in set(df["exp_id"].astype(str))

def get_proximity_stats_from_results(exp_id):
    prox_id = f"{exp_id}_proximity"
    if not os.path.exists(RESULTS_CSV):
        return None
    try:
        df = pd.read_csv(RESULTS_CSV)
    except Exception:
        return None
    df = df[df["exp_id"].astype(str) == prox_id]
    if df.empty:
        return None
    return df.iloc[-1].to_dict()

def find_experiment_dirs(exp_id, roots=None):
    roots = roots or [DRIVE_BASE, KAGGLE_INPUT_ROOT]
    matches = []
    for root in roots:
        if not os.path.exists(root):
            continue
        for dirpath, _, _ in os.walk(root):
            if os.path.basename(dirpath) == exp_id and os.path.basename(os.path.dirname(dirpath)) == "adapters":
                matches.append(dirpath)
    def _rank(path):
        norm = path.replace("\\", "/")
        score = 0
        if norm.startswith(DRIVE_BASE):
            score += 10
        elif norm.startswith(KAGGLE_INPUT_ROOT):
            score += 5
        return score
    return sorted(set(matches), key=_rank, reverse=True)

def find_latest_checkpoint(exp_id):
    candidates = []
    for exp_dir in find_experiment_dirs(exp_id):
        if not os.path.isdir(exp_dir):
            continue
        for name in os.listdir(exp_dir):
            if name.startswith("checkpoint-"):
                suffix = name.split("-")[-1]
                if suffix.isdigit():
                    candidates.append((int(suffix), os.path.join(exp_dir, name)))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1]

# ── Model loader (4-bit NF4 QLoRA) ───────────────────────────────────────────
def load_model_4bit(model_id):
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb,
        device_map={"": 0},
        trust_remote_code=True,
    )
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    model.config.use_cache = False
    print(f"✅ Loaded {model_id}")
    return model, tok

# ── MMLU evaluator ────────────────────────────────────────────────────────────
def eval_mmlu_subject(model, tok, subject, max_samples=None):
    try:
        ds = load_dataset("cais/mmlu", subject, split="test")
    except Exception as e:
        print(f"  ⚠️ {subject}: {e}")
        return float("nan")
    if max_samples:
        ds = ds.select(range(min(max_samples, len(ds))))
    correct, total = 0, 0
    label_map = {0:"A", 1:"B", 2:"C", 3:"D"}
    for ex in ds:
        choices_str = "\n".join([f"{l}) {c}" for l, c in zip("ABCD", ex["choices"])])
        prompt = (
            f"The following is a multiple choice question. "
            f"Answer with only the letter A, B, C, or D.\n\n"
            f"Question: {ex['question']}\n{choices_str}\n\nAnswer:"
        )
        messages = [{"role": "user", "content": prompt}]
        try:
            text = tok.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            messages_nothink = [{"role": "user", "content": prompt + " /no_think"}]
            try:
                text = tok.apply_chat_template(
                    messages_nothink, tokenize=False, add_generation_prompt=True
                )
            except Exception:
                text = prompt
        ids = tok(text, return_tensors="pt").input_ids.to(model.device)
        with torch.no_grad():
            out = model.generate(
                ids,
                max_new_tokens=5,
                do_sample=False,
                pad_token_id=tok.eos_token_id,
                eos_token_id=tok.eos_token_id,
            )
        gen_text = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip().upper()
        match = re.search(r"[ABCD]", gen_text)
        pred = match.group(0) if match else "X"
        correct += int(pred == label_map[ex["answer"]])
        total += 1
    return correct / total if total > 0 else float("nan")

def eval_all_mmlu(model, tok, max_samples=None):
    results = {}
    for i, subj in enumerate(MMLU_SUBJECTS):
        acc = eval_mmlu_subject(model, tok, subj, max_samples=max_samples)
        results[subj] = acc
        if (i + 1) % 5 == 0 or (i + 1) == len(MMLU_SUBJECTS):
            print(f"  [{i+1:2d}/57] {subj}: {acc:.3f}")
    return results

# ── Semantic proximity ────────────────────────────────────────────────────────
_embedder = None
def get_embedder():
    global _embedder
    if _embedder is None:
        _embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
    return _embedder

def compute_subject_similarities(domain_key):
    emb = get_embedder()
    domain_vec = emb.encode(DOMAIN_DESCRIPTIONS[domain_key], normalize_embeddings=True)
    subject_texts = [s.replace("_", " ") for s in MMLU_SUBJECTS]
    subject_vecs = emb.encode(
        subject_texts,
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=False,
    )
    return {s: float(np.dot(domain_vec, v)) for s, v in zip(MMLU_SUBJECTS, subject_vecs)}

def run_proximity_test(domain_key, forgetting_dict, exp_id):
    sims = compute_subject_similarities(domain_key)
    valid = [s for s in MMLU_SUBJECTS if s in forgetting_dict and not np.isnan(forgetting_dict[s])]
    sim_vals = [sims[s] for s in valid]
    fgt_vals = [forgetting_dict[s] for s in valid]
    r, p = pearsonr(sim_vals, fgt_vals)
    prox = [forgetting_dict[s] for s in valid if s in MEDICAL_MMLU]
    dist = [forgetting_dict[s] for s in valid if s not in MEDICAL_MMLU]
    t_stat, t_p = ttest_ind(prox, dist) if (len(prox) > 1 and len(dist) > 1) else (float("nan"), float("nan"))
    pm, dm = np.mean(prox), np.mean(dist)
    direction = "proximal>distal (matches InternAL)" if pm < dm else "proximal<=distal (reversal)"
    result = {
        "domain": domain_key,
        "h1_pearson_r": round(r, 4),
        "h1_pearson_p": round(p, 6),
        "h3_t_stat": round(t_stat, 4) if not np.isnan(t_stat) else "nan",
        "h3_t_p": round(t_p, 6) if not np.isnan(t_p) else "nan",
        "h3_proximal_mean_forgetting": round(pm, 4) if not np.isnan(pm) else "nan",
        "h3_distal_mean_forgetting": round(dm, 4) if not np.isnan(dm) else "nan",
        "h3_direction": direction,
        "n_subjects": len(valid),
    }
    print(f"  H1 r={r:.3f} p={p:.5f} | H3: {direction}")
    return result

# ── LoRA fine-tuner ───────────────────────────────────────────────────────────
def run_finetune(base_model_id, train_dataset, exp_id, lora_rank=16, n_steps=500, seed=42):
    set_seed(seed)
    adapter_path = f"{DRIVE_BASE}/adapters/{exp_id}"
    os.makedirs(adapter_path, exist_ok=True)

    checkpoint_every = min(100, max(25, n_steps // 4 if n_steps >= 4 else 1))
    logging_every = max(10, min(50, max(1, n_steps // 10)))
    resume_checkpoint = find_latest_checkpoint(exp_id) if SEARCH_PREVIOUS_OUTPUTS else None

    print(f"\n🔧 {exp_id}: rank={lora_rank}, steps={n_steps}")
    print(f"   💾 Checkpoints every {checkpoint_every} steps")
    if resume_checkpoint:
        print(f"   🔁 Resuming from: {resume_checkpoint}")

    model, tok = load_model_4bit(base_model_id)
    lora_cfg = LoraConfig(
        r=lora_rank,
        lora_alpha=lora_rank * 2,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    trainer = SFTTrainer(
        model=model,
        processing_class=tok,
        train_dataset=train_dataset,
        args=SFTConfig(
            output_dir=adapter_path,
            max_steps=n_steps,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=min(50, n_steps//10),
            learning_rate=2e-4,
            fp16=False,
            bf16=True,
            logging_steps=50,
            save_strategy="steps",
            save_steps=100,
            dataset_text_field="text",
            max_length=512,
            seed=seed,
            report_to="none"
        )
    )

    print(f"🚀 Training {n_steps} steps ...")
    if resume_checkpoint:
        trainer.train(resume_from_checkpoint=resume_checkpoint)
    else:
        trainer.train()

    model.save_pretrained(adapter_path)
    tok.save_pretrained(adapter_path)
    print(f"💾 Adapter saved: {adapter_path}")
    return model, tok

print("✅ All utilities ready")


## Step 4 — Load Domain Control Datasets

### GSM8K — Grade School Math

**GSM8K (Cobbe et al., 2021)** is a dataset of ~8 500 linguistically diverse grade-school mathematics word problems with step-by-step solutions. It is the canonical benchmark for testing LLM mathematical reasoning.

For our purposes it serves as a **domain control**: if fine-tuning on math causes forgetting that correlates with *math*-proximate MMLU subjects (e.g. `college_mathematics`, `high_school_statistics`) but not medical ones, this would confirm H1 as a domain-general phenomenon rather than a medical artefact.

The training format is:
```
Question: <math word problem>
Answer: <step-by-step solution with final number>
```

### CodeAlpaca — Programming Instructions

**CodeAlpaca** is a 20k instruction-following dataset for code generation tasks (writing functions, debugging, explaining algorithms). It covers Python, JavaScript, SQL, and general programming concepts.

As the most semantically distant domain from medicine in this study, CodeAlpaca provides the strongest test of H1. If fine-tuning on code also produces proximity-structured forgetting (with code-proximate MMLU subjects like `college_computer_science`, `high_school_computer_science` most affected), this would be a compelling generalisation of the proximity effect.

### Why load both upfront?

Both datasets are small enough (~4 MB each) to load into RAM once and reuse. Loading them here avoids redundant downloads during E05 and E05b.


In [ ]:
# --- CELL 4: Load domain datasets ---
def get_gsm8k(n_total=4000, seed=42):
    print("⬇️  Loading GSM8K ...")
    ds = load_dataset("gsm8k", "main", split="train")
    df = ds.to_pandas().sample(min(n_total, len(ds)), random_state=seed)
    rows = [{"text": f"Question: {r['question']}\nAnswer: {r['answer']}"} for _, r in df.iterrows()]
    result = Dataset.from_list(rows)
    print(f"✅ GSM8K: {len(result)} examples")
    return result

def get_code_alpaca(n_total=4000, seed=42):
    print("⬇️  Loading CodeAlpaca ...")
    ds = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
    df = ds.to_pandas().sample(min(n_total, len(ds)), random_state=seed)
    rows = [{"text": f"Instruction: {r['instruction']}\nResponse: {r['output']}"} for _, r in df.iterrows()]
    result = Dataset.from_list(rows)
    print(f"✅ CodeAlpaca: {len(result)} examples")
    return result

selected_domain_controls = set(RUN_DOMAIN_CONTROL_EXPERIMENTS)
gsm8k_train = None
codealpaca_train = None

if "E05" in selected_domain_controls:
    gsm8k_train = get_gsm8k(n_total=DOMAIN_DATASET_SIZE, seed=SEED_PRIMARY)
else:
    print("⏭️ Skipping GSM8K load because E05 is not selected in Cell 1.")

if "E05b" in selected_domain_controls:
    codealpaca_train = get_code_alpaca(n_total=DOMAIN_DATASET_SIZE, seed=SEED_PRIMARY)
else:
    print("⏭️ Skipping CodeAlpaca load because E05b is not selected in Cell 1.")

if not selected_domain_controls:
    print("ℹ️ No domain-control experiments selected in Cell 1.")


---
## EXPERIMENT E05 — Domain Control: GSM8K (Mathematics)
⏱️ ~2.5 hours

### What this experiment tests

E05 replicates the E02 (MedQA, r=16, 500 steps) protocol exactly — except the training domain is **mathematics** instead of medicine. The design is fully controlled:

| Variable | Value |
|----------|-------|
| Model | Qwen3.5-9B |
| LoRA rank | 16 |
| Training steps | 500 |
| Seed | 42 |
| **Training data** | **GSM8K (math)** ← only difference |

### Expected results under H1

The domain description vector for GSM8K emphasises: *"mathematics arithmetic algebra word problems numerical computation"*. MMLU subjects with high cosine similarity to this description include `college_mathematics`, `high_school_mathematics`, `elementary_mathematics`, `high_school_statistics`, and `econometrics`.

If H1 holds:
- Pearson *r* should be positive (more math-similar MMLU subjects → more forgetting)
- The pattern of *which* subjects are most forgotten should differ from MedQA (math subjects are hit, not medical ones)

If H1 does *not* hold for math:
- Forgetting might be uniform across all subjects (no proximity structure)
- Or forgetting might be near-zero (math is too different from general MMLU for LoRA to cause meaningful drift)

### What `run_domain_experiment` does

1. Calls `run_finetune` with the given dataset and hyperparameters
2. Re-evaluates all 57 MMLU subjects on the fine-tuned model
3. Computes per-subject forgetting (Δacc)
4. Runs `run_proximity_test` for both H1 and H3
5. Logs all results to the master CSV
6. Clears VRAM before returning


In [ ]:
# --- CELL 5: E05 (domain control: math) ---
def run_domain_experiment(exp_id, domain_key, train_data, rank=16, n_steps=500, seed=42):
    print(f"\n{'='*60}")
    print(f"▶  {exp_id}: domain={domain_key}, rank={rank}, steps={n_steps}")
    print(f"{'='*60}")

    ft_model, ft_tok = run_finetune(MODEL_8B, train_data, exp_id, rank, n_steps, seed)
    ft_model.eval()

    print(f"\n📊 Re-evaluating MMLU after {exp_id} ...")
    post_accs = eval_all_mmlu(ft_model, ft_tok, max_samples=MMLU_MAX_SAMPLES)

    forgetting = {}
    for subj in MMLU_SUBJECTS:
        base_acc = baseline_accs.get(subj, float("nan"))
        post_acc = post_accs.get(subj, float("nan"))
        delta = post_acc - base_acc
        forgetting[subj] = delta
        log_result(exp_id, {
            "model": MODEL_8B, "domain": domain_key, "lora_rank": rank,
            "n_steps": n_steps, "replay_size": 0, "seed": seed,
            "subject": subj,
            "accuracy":  round(post_acc, 4) if not np.isnan(post_acc) else "nan",
            "is_medical": subj in MEDICAL_MMLU,
            "delta_acc": round(delta, 4)    if not np.isnan(delta)    else "nan",
            "forgetting": round(-delta, 4)  if not np.isnan(delta)    else "nan",
        })

    prox_stats = run_proximity_test(domain_key, forgetting, exp_id)
    log_result(f"{exp_id}_proximity", {
        "model": MODEL_8B, "domain": domain_key, "lora_rank": rank,
        "n_steps": n_steps, "replay_size": 0, "seed": seed,
        "subject": "ALL_PROXIMITY_STATS", **prox_stats,
        "delta_acc": 0, "forgetting": 0, "accuracy": 0, "is_medical": False,
    })

    mean_fgt = -np.nanmean(list(forgetting.values()))
    print(f"\n📋 {exp_id} SUMMARY")
    print(f"   Domain: {domain_key} | Mean forgetting: {mean_fgt:+.3f}")
    print(f"   H1 Pearson r: {prox_stats['h1_pearson_r']}  p: {prox_stats['h1_pearson_p']}")
    print(f"   H3 direction: {prox_stats['h3_direction']}")

    del ft_model, ft_tok
    torch.cuda.empty_cache()
    import gc; gc.collect()
    print("   🗑️  VRAM cleared")
    return forgetting, prox_stats

domain_results = globals().get("domain_results", {})

if "E05" in set(RUN_DOMAIN_CONTROL_EXPERIMENTS):
    if SKIP_COMPLETED_EXPERIMENTS and experiment_already_logged("E05"):
        print("⏭️ E05 already exists in all_results.csv; skipping training.")
        prox_existing = get_proximity_stats_from_results("E05")
        if prox_existing:
            domain_results["E05"] = {"proximity": prox_existing}
    else:
        fgt_E05, prox_E05 = run_domain_experiment("E05", "gsm8k", gsm8k_train)
        domain_results["E05"] = {"forgetting": fgt_E05, "proximity": prox_E05}
else:
    print("⏭️ Skipping E05. Add 'E05' to RUN_DOMAIN_CONTROL_EXPERIMENTS in Cell 1 to run it.")


---
## EXPERIMENT E05b — Domain Control: CodeAlpaca (Programming)
⏱️ ~2.5 hours

### What this experiment adds

CodeAlpaca completes the **three-domain comparison** required to evaluate H1 as a general effect:

| Domain | MMLU subjects most similar | Expected proximity pattern |
|--------|---------------------------|---------------------------|
| MedQA  | anatomy, clinical\_knowledge, medical\_genetics | Medical subjects most forgotten |
| GSM8K  | college\_mathematics, high\_school\_statistics | Math subjects most forgotten |
| CodeAlpaca | college\_computer\_science, high\_school\_computer\_science | CS subjects most forgotten |

If all three show significant *r* with the appropriate domain's similarity profile, H1 is a **robust, domain-general property** of LoRA fine-tuning — not a medical-language artefact.

### The domain comparison cell that follows

After E05b completes, we print a 3-row table comparing H1 Pearson *r* across all domains:

```
MedQA  H1 r: X.XXX  (from Notebook A, E02)
GSM8K  H1 r: X.XXX  (E05)
Code   H1 r: X.XXX  (E05b)
```

If all three are positive and significant (*p* < 0.05), this is strong evidence for H1 as a general phenomenon. If only MedQA shows it, H1 may be domain-specific.

> **Note on the H3 test here:** H3 uses the `MEDICAL_MMLU` partition as the "proximal" group — this is intentional. For the math and code domains, we *expect* H3 to show `proximal<=distal` (i.e., medical subjects are *not* disproportionately forgotten when fine-tuning on math or code). This negative result for H3 in non-medical domains is itself informative: it validates that the H3 medical-proximity effect in E01–E04 is domain-specific, not an artefact of the MMLU evaluation procedure.


In [ ]:
# --- CELL 6: E05b (code domain) ---
domain_results = globals().get("domain_results", {})

if "E05b" in set(RUN_DOMAIN_CONTROL_EXPERIMENTS):
    if SKIP_COMPLETED_EXPERIMENTS and experiment_already_logged("E05b"):
        print("⏭️ E05b already exists in all_results.csv; skipping training.")
        prox_existing = get_proximity_stats_from_results("E05b")
        if prox_existing:
            domain_results["E05b"] = {"proximity": prox_existing}
    else:
        fgt_E05b, prox_E05b = run_domain_experiment("E05b", "codealpaca", codealpaca_train)
        domain_results["E05b"] = {"forgetting": fgt_E05b, "proximity": prox_E05b}
else:
    print("⏭️ Skipping E05b. Add 'E05b' to RUN_DOMAIN_CONTROL_EXPERIMENTS in Cell 1 to run it.")

if domain_results:
    print("\n✅ DOMAIN CONTROL STATUS")
    print("\n📊 Domain comparison (H1 test):")
    print("  MedQA  H1 r: (see E02 in Notebook A results)")
    if "E05" in domain_results and domain_results["E05"].get("proximity"):
        prox = domain_results["E05"]["proximity"]
        print(f"  GSM8K  H1 r: {prox.get('h1_pearson_r', 'NA')}  p={prox.get('h1_pearson_p', 'NA')}")
    if "E05b" in domain_results and domain_results["E05b"].get("proximity"):
        prox = domain_results["E05b"]["proximity"]
        print(f"  Code   H1 r: {prox.get('h1_pearson_r', 'NA')}  p={prox.get('h1_pearson_p', 'NA')}")
else:
    print("\nℹ️ No domain-control experiments were run or loaded in this session.")


---
## Steps Curve Setup — Loading MedQA

### Hypothesis H4 — Does forgetting accumulate gradually or sharply?

**Contradiction #8** in the literature:

| Claim | Source | Mechanism proposed |
|-------|--------|--------------------|
| Forgetting onsets sharply in first ~200 steps, then plateaus | Zhai et al. | Early weight updates overwrite the most general subspaces; later steps only refine domain-specific subspaces |
| Forgetting grows gradually and monotonically with steps | Luo et al. | Accumulated gradient updates continuously erode general knowledge throughout training |

These are not just quantitatively different — they have **different mechanistic implications**:

- **Zhai (sharp onset):** suggests there is a critical early phase where catastrophic forgetting happens. Mitigations should focus on the first ~200 steps (e.g., reduced LR warmup, early stopping, selective layer freezing).
- **Luo (gradual):** suggests forgetting is proportional to total training duration. The safest mitigation is simply fewer steps — quality-of-fit vs. forgetting is a smooth trade-off.

### Experimental design

We run the same E02 setup (rank=16, MedQA, seed=42) at 3 additional step counts:

```
E06:  100 steps  (~1.5 h)
E07:  200 steps  (~2.0 h)
E02:  500 steps  ← already in Notebook A results
E08: 1000 steps  (~3.3 h)
```

All results end up in the shared `all_results.csv`, which Notebook D's analysis cell assembles into the full 4-point curve.

### Why MedQA is reloaded here

Notebook B loads GSM8K and CodeAlpaca in Cell 4, but MedQA is only needed for the steps curve. Loading it here keeps the memory footprint lower during the domain control experiments (Cells 5 and 6), which may matter on a RAM-constrained T4 session.


In [ ]:
# --- CELL 7: Load MedQA for steps curve ---
def get_stratified_medqa(n_total=4000, seed=42):
    print("⬇️  Loading MedQA ...")
    ds = load_dataset("GBaker/MedQA-USMLE-4-options", split="train")
    df = ds.to_pandas()

    def get_options(row):
        if "option_0" in df.columns:
            return [row[f"option_{i}"] for i in range(4)]
        elif "options" in df.columns:
            opts = row["options"]
            if isinstance(opts, dict):
                return [opts[k] for k in sorted(opts.keys())[:4]]
            elif isinstance(opts, list):
                return opts[:4]
        return ["A", "B", "C", "D"]

    if "meta_info" in df.columns and df["meta_info"].nunique() > 1:
        n_per_cat = max(500, n_total // df["meta_info"].nunique())
        sampled = df.groupby("meta_info", group_keys=False).apply(
            lambda x: x.sample(min(len(x), n_per_cat), random_state=seed)
        )
    else:
        sampled = df.sample(min(n_total, len(df)), random_state=seed)

    sampled = sampled.sample(frac=1, random_state=seed).reset_index(drop=True)
    rows = []
    for _, row in sampled.iterrows():
        opts = get_options(row)
        opts_str = "\n".join([f"{l}) {c}" for l, c in zip("ABCD", opts)])
        answer = str(row.get("answer", "A")).strip().upper()
        if answer not in "ABCD":
            answer = "A"
        rows.append({"text": f"Question: {row['question']}\n{opts_str}\nAnswer: {answer}"})

    result = Dataset.from_list(rows)
    print(f"✅ MedQA: {len(result)} examples")
    return result

if RUN_STEPS_EXPERIMENTS:
    medqa_train = get_stratified_medqa(n_total=MEDQA_DATASET_SIZE, seed=SEED_PRIMARY)
else:
    medqa_train = None
    print("⏭️ Skipping MedQA load because RUN_STEPS_EXPERIMENTS is empty.")


---
## EXPERIMENTS E06–E08 — Steps Curve (H4)
Fine-tune on MedQA at 100, 200, 1000 steps with r=16 fixed.  
**Combined with E02 (500 steps) from Notebook A → 4-point curve.**  
⏱️ ~6.8 GPU-hours total

### Reading the summary table

The H4 summary table at the bottom of this cell will look like:

```
Exp    Steps   Mean fgt        Med fgt
----------------------------------------
E06      100   ±X.XXXX         ±X.XXXX
E07      200   ±X.XXXX         ±X.XXXX
E08     1000   ±X.XXXX         ±X.XXXX
```

When you combine this with E02 (500 steps) from `all_results.csv`, you get the full curve. Plot `mean_fgt` against `steps` in Notebook D to visualise the shape:

- **Concave curve (rapid early rise then plateau)** → supports Zhai
- **Linear or convex curve** → supports Luo

### Why medical forgetting is reported separately

`Med fgt` tracks forgetting averaged over only the 9 medical MMLU subjects. If H3 holds, `Med fgt` should consistently exceed `Mean fgt` — i.e., medical knowledge erodes faster than general knowledge regardless of training duration. This would be further evidence for domain-proximate forgetting (H1/H3) being independent of total training time (H4).

### VRAM management between runs

Each iteration:
1. Loads a fresh base model (no carry-over from the previous run)
2. Trains and evaluates
3. `del ft_model, ft_tok` + `torch.cuda.empty_cache()` + `gc.collect()` to reclaim VRAM

Without step 3, the T4's 15 GB would be exhausted partway through the loop. The ~30-second overhead per run is worth the stability.


In [ ]:
# --- CELL 8: Steps curve experiments ---
selected_steps = list(RUN_STEPS_EXPERIMENTS)

if not selected_steps:
    print("⏭️ No steps experiments selected. Edit RUN_STEPS_EXPERIMENTS in Cell 1 to run E06/E07/E08.")
else:
    steps_results = {}

    for exp_id, n_steps in selected_steps:
        print(f"\n{'='*60}")
        print(f"▶  {exp_id}: steps={n_steps}, rank=16, domain=medqa")
        print(f"{'='*60}")

        if SKIP_COMPLETED_EXPERIMENTS and experiment_already_logged(exp_id):
            print(f"⏭️ {exp_id} already exists in all_results.csv; skipping training.")
            continue

        ft_model, ft_tok = run_finetune(
            MODEL_8B, medqa_train, exp_id, lora_rank=16, n_steps=n_steps, seed=SEED_PRIMARY
        )
        ft_model.eval()

        print(f"\n📊 Re-evaluating MMLU after {exp_id} ...")
        post_accs = eval_all_mmlu(ft_model, ft_tok, max_samples=MMLU_MAX_SAMPLES)

        forgetting = {}
        for subj in MMLU_SUBJECTS:
            base_acc = baseline_accs.get(subj, float("nan"))
            post_acc = post_accs.get(subj, float("nan"))
            delta = post_acc - base_acc
            forgetting[subj] = delta
            log_result(exp_id, {
                "model": MODEL_8B, "domain": "medqa", "lora_rank": 16,
                "n_steps": n_steps, "replay_size": 0, "seed": SEED_PRIMARY,
                "subject": subj,
                "accuracy":  round(post_acc, 4) if not np.isnan(post_acc) else "nan",
                "is_medical": subj in MEDICAL_MMLU,
                "delta_acc": round(delta, 4)    if not np.isnan(delta)    else "nan",
                "forgetting": round(-delta, 4)  if not np.isnan(delta)    else "nan",
            })

        steps_results[exp_id] = forgetting
        mean_fgt = -np.nanmean(list(forgetting.values()))
        med_fgt  = -np.nanmean([forgetting[s] for s in MEDICAL_MMLU if s in forgetting])
        print(f"\n📋 {exp_id}: steps={n_steps} → mean fgt={mean_fgt:+.4f}  med fgt={med_fgt:+.4f}")

        del ft_model, ft_tok
        torch.cuda.empty_cache()
        import gc; gc.collect()
        print("   🗑️  VRAM cleared")

    print("\n" + "=" * 60)
    print("✅ STEPS CURVE STATUS")
    print("=" * 60)
    print("\n📊 H4 — Steps curve (combine with E02 from Notebook A for the full 4-point curve):")
    print(f"{'Exp':6} {'Steps':7} {'Mean fgt':12} {'Med fgt':12}")
    print("-" * 40)

    df_all = pd.read_csv(RESULTS_CSV)
    for exp_id, n_steps in selected_steps:
        df_e = df_all[df_all["exp_id"] == exp_id].copy()
        df_e = df_e[pd.to_numeric(df_e["forgetting"], errors="coerce").notna()]
        if df_e.empty:
            print(f"{exp_id:6} {n_steps:7} {'not run':>12} {'not run':>12}")
            continue
        df_e["forgetting_num"] = pd.to_numeric(df_e["forgetting"])
        mf = df_e["forgetting_num"].mean()
        mmf = df_e[df_e["is_medical"] == True]["forgetting_num"].mean()
        print(f"{exp_id:6} {n_steps:7} {mf:+12.4f} {mmf:+12.4f}")

    print(f"\n✅ Results saved to: {RESULTS_CSV}")
    print("   Save the notebook version after this finishes so you can reuse the outputs later.")
    print("   Hand this file to the analysis notebook (Notebook D) when all experiments are complete.")


---
## Optional — Package outputs for reuse
Run the next cell if you want a single zip file containing `results/` and `adapters/`.


In [ ]:
# --- CELL 9: Package outputs for download / reuse ---
archive_base = f"{KAGGLE_WORKING_ROOT}/notebook_B_kaggle_outputs"
archive_path = shutil.make_archive(archive_base, "zip", DRIVE_BASE)
print(f"✅ Output bundle created: {archive_path}")
print("Create a Kaggle dataset from this zip or use Save Version to reuse checkpoints/results later.")
